# 04 Batch and parallel extraction

`extract_features_from_folder` and the `afx` command line run one config over every audio file below a folder and write one table. This notebook covers output formats, failed files and `AFXWarning`, the provenance record every output carries, dataset statistics for scaling, `n_jobs` and the CLI. Needs `pip install "afx-audio[parquet,sklearn]"`.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import afx_audio as afx
import demo_data

SR = 16000
tmp = Path(tempfile.mkdtemp(prefix='afx_batch_'))
audio_dir = tmp / 'audio'

## A folder of clips

Up to 24 files from the configured dataset, otherwise 24 synthetic clips in one subfolder per class. One file is deliberately unreadable.

In [ ]:
files = demo_data.find_audio(max_files=24)
if files:
    audio_dir.mkdir()
    for f in files:
        shutil.copy(f, audio_dir / os.path.basename(f))
else:
    demo_data.write_synthetic_folder(str(audio_dir), n_per_class=6, seconds=2.0, sr=SR)
(audio_dir / 'broken.wav').write_bytes(b'not a wav file')

n_files = sum(1 for p in audio_dir.rglob('*.wav'))
print(f'{n_files} .wav files under {audio_dir}')

## One call, one table

The config can be a dict (or a preset), a JSON path, or the bundled default. Keys of the result are paths relative to the folder, with forward slashes on every OS.

In [ ]:
config = {
    'sample_rate': SR,
    'features': {
        'mfcc': {'n_mfcc': 13, 'frame_size': 1024, 'hop_length': 512},
        'spectral_centroid': {'frame_size': 1024, 'hop_length': 512},
        'spectral_flatness': {'frame_size': 1024, 'hop_length': 512},
        'zcr': {'frame_size': 1024, 'hop_length': 512},
        'hnr': {'frame_size': 1024, 'hop_length': 512},
    },
    'aggregation': 'mean',
}
results = afx.extract_features_from_folder(str(audio_dir), config, str(tmp / 'features.csv'), verbose=False)
ok = [k for k in results if k != 'errors']
print(len(ok), 'files extracted; first key:', ok[0])
print('errors:', results.get('errors'))

## Failed files and warnings

A file that cannot be read or processed never stops the batch. Its error goes to `<output>.errors.json` and to the `'errors'` entry of the returned dict, and is never mixed into the feature table.

In [ ]:
print(json.loads((tmp / 'features.csv.errors.json').read_text()))

The summary line above was an `afx.AFXWarning`. Every warning AFX emits has that category (or its subclass `AFXDeprecationWarning`), so a filter on it silences AFX without hiding warnings from NumPy, SciPy or your own code, and `'error'` turns AFX warnings into exceptions in a test suite.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    afx.extract_features_from_folder(str(audio_dir), config, str(tmp / 'again.csv'), verbose=False)
print([(w.category.__name__, str(w.message)[:40]) for w in caught])

with warnings.catch_warnings():
    warnings.filterwarnings('error', category=afx.AFXWarning)
    try:
        afx.extract_features_from_folder(str(audio_dir), config, str(tmp / 'again.csv'), verbose=False)
    except afx.AFXWarning as e:
        print('raised:', e)

## Output formats

The format follows the output extension (`save_format=` or `afx --format` override it).

| extension | content | load with |
|---|---|---|
| `.csv` | one row per file, named columns | `pd.read_csv(path, index_col='file')` |
| `.parquet` | same as CSV, typed; needs `afx-audio[parquet]` | `pd.read_parquet(path)` |
| `.json` | `{file: {feature: list}}`, NaN as `null` | `json.load` |
| `.npy` | pickled dict of arrays | `np.load(path, allow_pickle=True).item()` |

In [ ]:
warnings.filterwarnings('ignore', category=afx.AFXWarning)  # broken.wav, reported above
for ext in ('csv', 'parquet', 'json', 'npy'):
    afx.extract_features_from_folder(str(audio_dir), config, str(tmp / f'features.{ext}'), verbose=False)

df = pd.read_csv(tmp / 'features.csv', index_col='file')
pq = pd.read_parquet(tmp / 'features.parquet')
js = json.loads((tmp / 'features.json').read_text())
npy = np.load(tmp / 'features.npy', allow_pickle=True).item()

print('csv    ', df.shape, list(df.columns[:3]), '...')
print('parquet', pq.shape)
print('json   ', len(js), 'files;', {k: len(v) for k, v in next(iter(js.values())).items()})
print('npy    ', len(npy), 'files;', {k: len(v) for k, v in next(iter(npy.values())).items()})
np.testing.assert_allclose(df.to_numpy(), pq.loc[df.index].to_numpy())

Column names match `afx.flatten_features` and `AFXFeatureExtractor.get_feature_names_out()`, so a table written here lines up with features computed in memory.

In [ ]:
signal, sr = afx.load_audio(audio_dir / df.index[0], sr=SR)
names, vector = afx.flatten_features(afx.extract_all_features(signal, sr, config))
assert names == list(df.columns)
np.testing.assert_allclose(vector, df.iloc[0].to_numpy(), equal_nan=True)
df.head()

## Provenance

Every output records how it was made: the validated config and its `config_hash`, the preset id, AFX, NumPy, SciPy and Python versions, the sample rate and resampling, and the sha256 of each input file. Parquet keeps the record in its schema metadata; the other formats write `<output>.provenance.json`. `afx.read_provenance` reads either.

In [ ]:
record = afx.read_provenance(tmp / 'features.csv')
print('sidecar        :', (tmp / 'features.csv.provenance.json').exists())
for key in ('afx_version', 'config_hash', 'preset_id', 'preset_modified', 'sample_rate', 'n_files', 'n_errors'):
    print(f'{key:15s}: {record[key]}')
print('versions       :', record['versions'])
print('resampling     :', record['resampling'])

The hash is over the config's canonical JSON (sorted keys), so it identifies the feature definition independently of key order and of where the file came from. Parquet carries the same record.

In [ ]:
assert afx.config_hash(record['config']) == record['config_hash']
assert afx.read_provenance(tmp / 'features.parquet')['config_hash'] == record['config_hash']
reordered = dict(reversed(list(record['config'].items())))
print('reordered config, same hash:', afx.config_hash(reordered) == record['config_hash'])

first, digest = next(iter(record['file_sha256'].items()))
print(len(record['file_sha256']), 'file hashes;', first, digest[:16], '...')
assert hashlib.sha256((audio_dir / first).read_bytes()).hexdigest() == digest

## Dataset statistics

`dataset_stats=True` (`afx --stats`) also writes `<output>.stats.json`: per-column count, mean, variance, min, max and NaN count, accumulated while extracting, so the table never has to be loaded. `afx.load_stats` reads it and `afx_audio.sklearn.scaler_from_stats` turns it into a fitted `StandardScaler` for data extracted later with the same config.

In [ ]:
from afx_audio.sklearn import scaler_from_stats
from sklearn.preprocessing import StandardScaler

afx.extract_features_from_folder(str(audio_dir), config, str(tmp / 'train.parquet'), dataset_stats=True,
                                 verbose=False)
stats = afx.load_stats(tmp / 'train.parquet')
print(stats['n_rows'], 'rows,', len(stats['columns']), 'columns; NaN cells:', int(stats['nan_count'].sum()))
print(pd.DataFrame({k: stats[k][:4] for k in ('mean', 'std', 'min', 'max')}, index=stats['columns'][:4]))

scaler = scaler_from_stats(stats)
train = pd.read_parquet(tmp / 'train.parquet')[stats['columns']]
reference = StandardScaler().fit(train.to_numpy())
np.testing.assert_allclose(scaler.mean_, reference.mean_)
np.testing.assert_allclose(scaler.scale_, reference.scale_)
print('equal to StandardScaler fitted on the table')

A new batch, extracted with the same config, is scaled with the training statistics.

In [ ]:
new_dir = tmp / 'new_batch'
demo_data.write_synthetic_folder(str(new_dir), n_per_class=3, seconds=2.0, sr=SR, seed=500)
afx.extract_features_from_folder(str(new_dir), config, str(tmp / 'new.parquet'), verbose=False)
new = pd.read_parquet(tmp / 'new.parquet')[stats['columns']]
assert afx.read_provenance(tmp / 'new.parquet')['config_hash'] == record['config_hash']

scaled = scaler.transform(new.to_numpy())
print(f'new batch {scaled.shape}: column means in [{np.nanmean(scaled, 0).min():.2f}, '
      f'{np.nanmean(scaled, 0).max():.2f}], stds in [{np.nanstd(scaled, 0).min():.2f}, {np.nanstd(scaled, 0).max():.2f}]')

## Serial and parallel

`n_jobs=N` processes files in `N` worker processes (`0` or negative means all cores). Output and order are identical to a serial run. Worker start-up is a fixed cost, so the speed-up approaches the core count only for long files or many of them. In a script, call it under `if __name__ == '__main__':` (see `03_long_recordings_and_custom_features.ipynb`).

In [ ]:
timings = {}
outputs = {}
for n_jobs in (1, 2):
    t0 = time.perf_counter()
    outputs[n_jobs] = afx.extract_features_from_folder(
        str(audio_dir), config, str(tmp / f'features_j{n_jobs}.npy'), n_jobs=n_jobs, verbose=False)
    timings[n_jobs] = time.perf_counter() - t0

assert list(outputs[1]) == list(outputs[2])
for key, feats in outputs[1].items():
    if key == 'errors':
        assert feats == outputs[2]['errors']
        continue
    for name, value in feats.items():
        np.testing.assert_array_equal(value, outputs[2][key][name])
print({f'n_jobs={k}': f'{v:.2f} s' for k, v in timings.items()}, f'on {os.cpu_count()} cores; outputs identical')

## The `afx` command

The installed `afx` command runs the same function. It is invoked here as `python -m afx_audio.cli` so the notebook uses the kernel's interpreter; in a shell write `afx ...`. Without `-q` it ends with the preset id and config hash; `--provenance FILE` prints an output's record as JSON.

In [ ]:
def afx_cli(*args, show=True):
    run = subprocess.run([sys.executable, '-m', 'afx_audio.cli', *args], capture_output=True, text=True)
    if show:
        print(f'$ afx {" ".join(args)}  (exit {run.returncode})')
        # drop the per-file "[i/n] Processing" lines
        lines = [s for s in (run.stdout + run.stderr).splitlines() if not s.startswith('[')]
        print('\n'.join(lines)[:1500], '\n')
    return run.stdout


afx_cli('--describe', 'spectral_flatness')
afx_cli('--list-presets')
afx_cli(str(audio_dir), '--preset', 'speech@1', '-o', str(tmp / 'speech.parquet'), '--stats')
afx_cli(str(audio_dir), '--features', 'mfcc,zcr', '--aggregation', 'summary', '-o', str(tmp / 'small.csv'), '-q')
print(pd.read_parquet(tmp / 'speech.parquet').shape, pd.read_csv(tmp / 'small.csv').columns[:5].tolist())

In [ ]:
cli_record = json.loads(afx_cli('--provenance', str(tmp / 'speech.parquet'), show=False))
print({k: cli_record[k] for k in ('preset_id', 'preset_modified', 'config_hash', 'n_files', 'n_errors')})
assert cli_record['config_hash'] == afx.config_hash(afx.load_preset('speech@1'))
print('stats columns:', len(afx.load_stats(tmp / 'speech.parquet')['columns']))

`-j N` sets `n_jobs`, `--coverage` adds `<feature>_valid_frac` columns and `--no-hash` skips the file hashes. `afx --help` lists every option; `--list-features` and `--list-presets` print the catalogues.

In [ ]:
shutil.rmtree(tmp)